# Cloud Optimized GeoTIFF: Overview (Pyramid) Generation

Cloud Optimized GeoTIFFs (COGs) store internal overview images at reduced
resolution, so map viewers and tiling servers can fetch the right zoom
level without downloading the full file.

xarray-spatial generates these overviews natively during `to_geotiff()`
with `cog=True`. No GDAL or `gdaladdo` post-processing required.

This notebook covers:
- Writing a COG with automatic overviews
- Choosing resampling methods
- Specifying explicit overview levels
- Verifying the result

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial.geotiff import open_geotiff, to_geotiff
from xrspatial.geotiff._header import parse_header, parse_all_ifds

## Create synthetic terrain data

A 256x256 elevation surface with some peaks, to give the overviews
something visually meaningful to downsample.

In [ ]:
rng = np.random.RandomState(42)
y = np.linspace(45.0, 44.0, 256)
x = np.linspace(-120.0, -119.0, 256)

# Smooth terrain from Gaussian peaks
xx, yy = np.meshgrid(x, y)
terrain = (
    800 * np.exp(-((xx + 119.5)**2 + (yy - 44.5)**2) / 0.02)
    + 600 * np.exp(-((xx + 119.3)**2 + (yy - 44.7)**2) / 0.05)
    + 100 * rng.rand(256, 256)
).astype(np.float32)

da = xr.DataArray(
    terrain, dims=['y', 'x'],
    coords={'y': y, 'x': x},
    attrs={'crs': 4326},
    name='elevation',
)

fig, ax = plt.subplots(figsize=(6, 5))
da.plot(ax=ax, cmap='terrain')
ax.set_title('Synthetic elevation')
plt.tight_layout()
plt.show()

## Write a COG with automatic overviews

Pass `cog=True` and xarray-spatial handles the rest: it halves the
dimensions until the smallest overview fits within a single tile,
then writes all levels into the file with IFDs at the start (the COG
layout requirement for efficient HTTP range requests).

In [ ]:
import tempfile, os

tmpdir = tempfile.mkdtemp(prefix='cog_demo_')
cog_path = os.path.join(tmpdir, 'auto_overviews.tif')

to_geotiff(da, cog_path, cog=True, compression='deflate')

# Check how many IFDs (resolution levels) were written
with open(cog_path, 'rb') as f:
    raw = f.read()

header = parse_header(raw)
ifds = parse_all_ifds(raw, header)
for i, ifd in enumerate(ifds):
    label = 'Full resolution' if i == 0 else f'Overview {i}'
    print(f'{label}: {ifd.width} x {ifd.height}')

## Resampling methods

Different data types need different resampling:
- **mean** (default): continuous data like elevation or temperature
- **nearest**: categorical or index data (land cover classes)
- **mode**: majority-class downsampling for classified rasters
- **min** / **max** / **median**: when you need conservative bounds

In [ ]:
methods = ['mean', 'nearest', 'min', 'max']
fig, axes = plt.subplots(1, len(methods), figsize=(14, 3))

for ax, method in zip(axes, methods):
    path = os.path.join(tmpdir, f'cog_{method}.tif')
    to_geotiff(da, path, cog=True, compression='deflate',
               overview_resampling=method)

    # Read back the first overview level
    result = open_geotiff(path, overview_level=1)
    result.plot(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(f'{method} ({result.shape[0]}x{result.shape[1]})')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle('Overview level 1 by resampling method', y=1.02)
plt.tight_layout()
plt.show()

## Explicit overview levels

For fine-grained control, pass `overview_levels` as a list of
decimation factors. Each factor halves the previous level once.

In [ ]:
explicit_path = os.path.join(tmpdir, 'explicit_levels.tif')
to_geotiff(da, explicit_path, cog=True, compression='deflate',
           overview_levels=[2, 4, 8])

with open(explicit_path, 'rb') as f:
    raw = f.read()

header = parse_header(raw)
ifds = parse_all_ifds(raw, header)
for i, ifd in enumerate(ifds):
    label = 'Full resolution' if i == 0 else f'Overview {i}'
    print(f'{label}: {ifd.width} x {ifd.height}')

## Verify round-trip

Full-resolution pixel values are preserved through the COG write.

In [ ]:
result = open_geotiff(cog_path)
max_diff = float(np.max(np.abs(result.values - terrain)))
print(f'Max pixel difference: {max_diff}')
assert max_diff < 1e-5, 'Values should round-trip exactly'

In [ ]:
# Clean up
import shutil
shutil.rmtree(tmpdir)